# IOAI — 2025 Team Selection Day3 Nlp Code Difficulty (Colab 자동 설정판)

아래 **설정 셀을 먼저 실행**하면 공개 데이터 소스에서 데이터를 받아 이 폴더에 `train.csv`/`test.csv` 등으로 준비합니다. 이후 셀이 그대로 학습/예측하고, 만들어진 제출 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

> 런타임 메뉴 → **런타임 유형 변경 → GPU** (필요 시).

In [ ]:
# === 데이터 자동 준비 (가장 먼저 실행) ===
import os
BASE='https://raw.githubusercontent.com/scvcoder/ioai-colab/main/data/2025-team-selection-day3-nlp-code-difficulty'
for f in ['train.csv','val.csv','test.csv','sample_submission.csv']:
    if not os.path.exists(f): os.system(f'wget -q {BASE}/{f}')
print('데이터 준비:', [f for f in ['train.csv','val.csv','test.csv','sample_submission.csv'] if os.path.exists(f)])
import os; print('작업 폴더:', os.getcwd()); print('내용:', sorted(os.listdir('.')))

# Day 3 — Code Difficulty Classification (Kazakhstan IOAI Team Selection 2025)

> **Kazakhstan – Team Selection 2025 · Day 3 (ML on text)**

Classify a source-code snippet as **easy / medium / hard**. Score = **weighted F1** over a held-out validation split. This baseline uses **TF-IDF + Logistic Regression** and writes `submission_val.csv` (graded) and `submission.csv` (Kaggle test format).

*Real-data notice:* original competition data from the public up-solving mirror `kaggle.com/competitions/tst-day-3-upsolving`. The hidden test is graded on Kaggle; here we self-grade on a held-out 20% validation split.

In [ ]:
import os, pandas as pd, numpy as np
def _find(f):
    for b in [".","..","../..","/home/yhpark/ioai/problems/Kazakhstan-2025/Team-Selection/Day3_NLP_Code_Difficulty"]:
        if os.path.exists(os.path.join(b,f)): return b
    return "."
DATA=_find("train.csv")
train=pd.read_csv(os.path.join(DATA,"train.csv"))   # id, difficulty, code
val  =pd.read_csv(os.path.join(DATA,"val.csv"))     # id, code   (held-out, graded by weighted F1)
test =pd.read_csv(os.path.join(DATA,"test.csv"))    # id, code   (Kaggle test)
print("train",train.shape,"val",val.shape,"test",test.shape,"| classes",sorted(train.difficulty.unique()))


## TF-IDF + Logistic Regression

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
vec=TfidfVectorizer(max_features=20000, ngram_range=(1,2), sublinear_tf=True)
Xtr=vec.fit_transform(train["code"].astype(str))
clf=LogisticRegression(max_iter=2000, C=4.0, class_weight="balanced")
clf.fit(Xtr, train["difficulty"].astype(str))


## Predict → `submission_val.csv` (graded) + `submission.csv` (Kaggle)

In [ ]:
pd.DataFrame({"id":val["id"], "difficulty":clf.predict(vec.transform(val["code"].astype(str)))}).to_csv("submission_val.csv",index=False)
pd.DataFrame({"id":test["id"],"difficulty":clf.predict(vec.transform(test["code"].astype(str)))}).to_csv("submission.csv",index=False)
print("wrote submission_val.csv / submission.csv")


## 제출 파일 모으기
아래 셀을 실행하면 제출 파일이 **최상위(`/content`)로 복사**되어 왼쪽 파일 탐색기에 바로 보입니다.
그 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

In [ ]:
# === 제출 파일을 /content 로 모으기 (마지막에 실행) ===
import os, glob, shutil
TARGETS = ['submission_val.csv']
OUT = "/content" if os.path.isdir("/content") else os.getcwd()
found = []
for name in TARGETS:
    hits = [name] if os.path.exists(name) else glob.glob(f"**/{name}", recursive=True)
    if not hits:
        print("아직 없음(해당 셀을 먼저 실행하세요):", name); continue
    dst = os.path.join(OUT, os.path.basename(hits[0]))
    if os.path.abspath(hits[0]) != os.path.abspath(dst):
        shutil.copy2(hits[0], dst)
    found.append(dst)
print("제출 파일 저장 위치(파일 탐색기 최상위):", found)